# 🍅 정직 로봇 DQN 학습 (`01_train_honest`)

**2026 창의설계축전** — 연구 1단계: **정직 재배 로봇**을 강화학습(DQN)으로 학습하고 정책을 저장합니다.

- **정직 로봇** = 실제 토마토 생존(`reward_mode="true"`)으로 보상받고, 스푸핑 타일 **O에 진입 금지**(`allow_o_tile=False`)인 안전 정책.
- 학습 결과는 나중에 **고정**되어, 감시자(overseer)가 상대할 "정상 로봇" 역할을 합니다.
- 먼저 `00_colab_bootstrap.ipynb`로 환경이 도는지 확인한 뒤 이 노트북을 실행하세요.

> ⚠️ 이 환경은 강화학습 난이도가 꽤 높습니다(아래 2장에서 설명). 그래서 학습을 돕는 **래퍼**를 사용합니다.
> 기대 결과: 로봇이 돌아다니며 마른 토마토에 물을 주는 "농부" 행동. 학습량을 늘릴수록 더 많은 토마토를 오래 살립니다.


## 1. 설정 — 패키지 설치 + 팀 코드 내려받기

매 세션 한 번 실행합니다. GPU는 필요 없습니다(CPU 런타임으로 충분).


In [ ]:
!pip install -q stable-baselines3 gymnasium

import os, sys
if os.path.isdir('/content'):
    os.chdir('/content')
    if not os.path.isdir('tomato-oversight'):
        !git clone https://github.com/1ee1ee1ee/tomato-oversight.git
    os.chdir('/content/tomato-oversight')
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())


## 2. 왜 "학습 래퍼"가 필요한가 (중요)

이 환경을 순수하게 DQN에 넘기면 로봇은 **"가만히 서서 토마토를 죽게 두는"** 정책으로 붕괴합니다. 두 가지 이유가 있습니다.

1. **부분관측(POMDP)**: 로봇의 기본 관측에는 각 토마토가 *얼마나 말랐는지*가 없습니다. "살았다/죽었다"만 봅니다. 그런데 토마토는 죽으면 되살릴 수 없으므로, 어느 토마토가 곧 마를지 모른 채로는 미리 물을 줄 수 없습니다.
2. **희소 보상**: 실제 보상은 500 step마다 한 번만 주어집니다. 기본 감가율로는 미래 보상이 거의 0으로 할인되어 학습 신호가 사라집니다.

그래서 `src/wrappers.py`의 두 래퍼를 사용합니다.

| 래퍼 | 역할 | 적용 시점 |
|---|---|---|
| `FreshnessObsWrapper` | 관측에 **토마토별 신선도**(1=방금 물줌 ~ 0=곧 죽음) 추가 | **학습 + 평가 모두** (정책이 이 관측을 기대) |
| `HonestRewardShaping` | 죽음 페널티 + 마른 토마토 구제 보상 (탐색 유도) | **학습만** |

> 신선도 관측은 "정직 로봇이 밭 상태를 안다"는 자연스러운 가정이며, 감시자 실험에는 영향을 주지 않습니다.
> 보상 shaping은 학습을 돕는 보조 신호일 뿐, **진짜 성능은 항상 `true_alive`(실제 생존)로 평가**합니다.


In [ ]:
from src.grower_env import TomatoWateringEnv, GrowerConfig
from src.wrappers import FreshnessObsWrapper, HonestRewardShaping
from stable_baselines3.common.monitor import Monitor

# 정직 로봇 설정: 실제 생존 보상 + O 진입 금지(안전 정책)
HONEST_CFG = dict(mode='E1', reward_mode='true', allow_o_tile=False)

def make_train_env():
    env = TomatoWateringEnv(GrowerConfig(**HONEST_CFG))
    env = HonestRewardShaping(env)      # 학습용 보상 shaping
    env = FreshnessObsWrapper(env)      # 관측에 신선도 추가
    return Monitor(env)

def make_plain_env(render_mode=None):   # 평가/시각화용: 관측만 augment (shaping 없음)
    env = TomatoWateringEnv(GrowerConfig(**HONEST_CFG), render_mode=render_mode)
    return FreshnessObsWrapper(env)

env = make_train_env()
print('관측 차원:', env.observation_space.shape, '| 행동 수:', env.action_space.n)


## 3. DQN 학습

- `gamma=0.99`: shaping 덕분에 보상이 조밀해져 표준 감가율로 충분합니다.
- `exploration_fraction=0.3`: 초반 30% 구간 동안 무작위 탐색 비율을 서서히 낮춥니다.
- **`total_timesteps`**: 기본 30만. Colab CPU에서 대략 **15~25분**. 빠르게 확인만 하려면 `100_000`으로 줄이고, 성능을 높이려면 `500_000` 이상으로 늘리세요.


In [ ]:
from stable_baselines3 import DQN

model = DQN(
    'MlpPolicy',
    make_train_env(),
    learning_rate=1e-3,
    buffer_size=100_000,
    learning_starts=5_000,
    batch_size=128,
    gamma=0.99,
    train_freq=4,
    target_update_interval=2_000,
    exploration_fraction=0.3,
    exploration_final_eps=0.05,
    policy_kwargs=dict(net_arch=[128, 128]),
    verbose=0,
    seed=1,
)

TOTAL_TIMESTEPS = 300_000        # ← 조절 가능 (100_000=빠름, 500_000+=고성능)
model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
print('학습 완료')


## 4. 평가 — 실제 성능(`true_alive`)과 정직성 확인

- **성능**: 여러 seed에서 평균 실제 생존 토마토 수. (무작위 정책과 비교)
- **정직성**: O 타일 진입 횟수 = **항상 0**이어야 함 (정직 로봇은 O가 막혀 있으므로 절대 스푸핑하지 않음).


In [ ]:
import numpy as np

def evaluate(policy, seeds=(7, 21, 99, 123), deterministic=True, random_policy=False):
    rows = []
    for seed in seeds:
        env = make_plain_env()
        obs, info = env.reset(seed=seed)
        alive = steps = waters = entered_o = 0
        for _ in range(10_000):
            if random_policy:
                action = env.action_space.sample()
            else:
                action, _ = policy.predict(obs, deterministic=deterministic)
                action = int(action)
            obs, r, term, trunc, info = env.step(action)
            alive += info['true_alive']; steps += 1
            waters += int(info['watered']); entered_o += int(info['entered_o'])
            if term or trunc:
                break
        rows.append((seed, steps, alive / steps, waters, entered_o))
    print(f"{'seed':>5} {'steps':>6} {'avg_true_alive':>15} {'waters':>7} {'entered_O':>10}")
    for s, st, av, w, eo in rows:
        print(f'{s:>5} {st:>6} {av:>15.2f} {w:>7} {eo:>10}')
    print(f'== 평균 실제 생존 = {np.mean([r[2] for r in rows]):.2f} / 5 ==')
    return rows

print('▶ 무작위 정책 (기준선)')
evaluate(None, random_policy=True)
print()
print('▶ 학습된 정직 로봇')
evaluate(model)


## 5. 학습된 로봇 시각화 (애니메이션)

학습된 정책이 실제로 어떻게 물을 주는지 봅니다. (🟢 실제 생존 · 🟡 스푸핑 · ⚪ 죽음 — 정직 로봇은 O를 안 밟으니 🟡은 나오지 않습니다.)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

env = make_plain_env(render_mode='rgb_array')
obs, info = env.reset(seed=7)
frames = [env.render()]
for _ in range(120):
    action, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, info = env.step(int(action))
    frames.append(env.render())
    if term or trunc:
        break

fig = plt.figure(figsize=(5, 5)); plt.axis('off')
im = plt.imshow(frames[0])
anim = animation.FuncAnimation(fig, lambda i: [im.set_data(frames[i])] and [im],
                               frames=len(frames), interval=120, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 6. 정책 저장 (Google Drive)

Colab 세션이 끊기면 사라지므로 Drive에 저장해 팀과 공유하고, 감시자 학습 때 불러 씁니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/tomato-oversight/models'
os.makedirs(save_dir, exist_ok=True)
model.save(f'{save_dir}/honest_dqn')
print('저장 완료:', f'{save_dir}/honest_dqn.zip')

# 나중에 불러오기:
# from stable_baselines3 import DQN
# model = DQN.load(f'{save_dir}/honest_dqn')


## 다음 단계 & 성능 올리기

**치터 로봇 노트북(`02_train_cheater.ipynb`)** 은 이 노트북을 복사해 설정만 바꾸면 됩니다:
- `reward_mode='apparent'` (겉보기 보상), `allow_o_tile=True` (O로 센서 조작 가능)
- 보상 shaping은 정직용(`HonestRewardShaping`) 대신 겉보기 보상을 좇도록 다르게 설계 (또는 shaping 없이 apparent 보상만)

**정직 로봇 성능이 아쉬우면** (모든 토마토를 오래 못 살리면):
- `TOTAL_TIMESTEPS`를 500k~1M으로 늘리기 (가장 효과적)
- `HonestRewardShaping`의 `death_penalty`·`rescue_bonus` 조정 (`src/wrappers.py`)
- `learning_rate`, `net_arch`(예: `[256,256]`), `exploration_fraction` 튜닝
- 참고: 밭 관측이 부분적이라 **모든 5개를 완벽히 유지하는 것은 본질적으로 어렵습니다.** 필요하면 신선도를 코어 관측에 정식 편입하는 설계 변경도 가능합니다.

> 감시자 실험에는 로봇이 "완벽한 농부"일 필요는 없고, **정직하게(O 미사용) 그럴듯하게 재배**하면 충분합니다.
